# 08 - Planning & Task Decomposition

## Scenario: Plan-and-Solve for Multi-Region Outages

Standard ReAct agents (Observe -> Decide -> Act) suffer from "ADHD". If you ask a ReAct agent to "Check the logs in EU, US, and AP regions and summarize them", it will often check EU, get distracted by a warning, try to fix it, and completely forget to check US and AP.

To fix this, we use the **Plan-and-Execute** architecture.
1. A **Planner Agent** takes the complex user request and breaks it down into an array of strict tasks (using Pydantic).
2. An **Executor Loop** iterates through the array one-by-one, executing the tasks systematically.

In this notebook, we'll build a planner for Northstar out-of-band investigations.

In [1]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field

# 1. Attempt to use the real API
if os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
else:
    # 2. Fallback to our local mock for students without keys
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    import sys
    import os
    sys.path.append(os.path.abspath("../../.."))
    from awsome_agents.mock_openai import MockOpenAI
    client = MockOpenAI()

# 3. Optional: Local LLMs
# If you prefer to use a local model like Llama 3 instead of the mock:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. The Planner

The planner has no tools. It only outputs a structured plan.

In [2]:
class PlanTask(BaseModel):
    task_id: int = Field(description="Sequential ID")
    instruction: str = Field(description="Specific, actionable instruction for the executor.")

class ExecutionPlan(BaseModel):
    tasks: list[PlanTask]

def generate_plan(user_request: str) -> ExecutionPlan:
    print("🧠 [Planner] Analyzing request and decomposing tasks...")
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a lead SRE. Break down the user's request into 3-4 concrete steps."},
                {"role": "user", "content": user_request}
            ],
            response_format=ExecutionPlan
        )
        return completion.choices[0].message.parsed
    except Exception:
        # Mock for local dev
        return ExecutionPlan(tasks=[
            PlanTask(task_id=1, instruction="Check EU region logs."),
            PlanTask(task_id=2, instruction="Check US region logs."),
            PlanTask(task_id=3, instruction="Compare latency between EU and US.")
        ])

plan = generate_plan("Please check the logs in EU and US and figure out which one is slower.")
for t in plan.tasks:
    print(f"  Task {t.task_id}: {t.instruction}")


🧠 [Planner] Analyzing request and decomposing tasks...
  Task 1: Check EU region logs.
  Task 2: Check US region logs.
  Task 3: Compare latency between EU and US.


## 2. The Executor

The executor is just a basic loop that iterates over the `plan.tasks` array. Because it is constrained to a single task at a time, it cannot get distracted.

In [3]:
def mock_execute_task(instruction: str):
    # In a real system, this would call your Agent Loop (from Module 02)
    print(f"🤖 [Executor] Executing: {instruction}")
    print("   ...done.\n")

for task in plan.tasks:
    mock_execute_task(task.instruction)

print("✅ [System] Entire plan executed successfully.")


🤖 [Executor] Executing: Check EU region logs.
   ...done.

🤖 [Executor] Executing: Check US region logs.
   ...done.

🤖 [Executor] Executing: Compare latency between EU and US.
   ...done.

✅ [System] Entire plan executed successfully.


## Checkpoint

**1. Why does the Plan-and-Execute architecture perform better than standard ReAct on long, complex tasks?**
- A) It uses a more expensive model.
- B) It forces the LLM to separate the "thinking/planning" phase from the "doing" phase, preventing it from getting distracted by intermediate tool outputs.
- C) It allows the LLM to skip tools entirely.
- D) It runs on a quantum computer.
